# 03 — Bipartite Networks & Gatekeepers (NYT)

Terza fase della pipeline: reti bipartite Autore-Keyword e Sezione-Keyword su `articles_ice`, proiezioni unipartite, estrazione dei "gatekeeper" editoriali (autori più centrali nella rete tematica) e confronto del vocabolario tra articoli signed/unsigned.

**Input**: `data/processed/nyt/articles_ice.parquet` (colonne `byline_json`, `keywords_json`, `section_name`).

Output prodotti:

- `data/processed/nyt/ice_bipartite_author_keyword.gexf`, `ice_author_projection.gexf`
- `data/processed/nyt/ice_bipartite_section_keyword.gexf`, `ice_section_projection.gexf`
- `data/processed/nyt/ice_gatekeepers.csv`, `ice_gatekeepers_by_period.csv`
- `data/processed/nyt/plots/{gatekeepers_barchart,volume_vs_influence,gatekeeper_heatmap,gatekeepers_by_period,vocab_diverging_bars}.png`
- `data/processed/nyt/vocab_signed_vs_unsigned.csv`
- `docs/insights_signed_vs_unsigned.md`



## Setup

In [ ]:
import html
import json
import logging
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from networkx.algorithms import bipartite

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"
DOCS_DIR = BASE_DIR / "docs"

IN_ICE = PROCESSED_DIR / "articles_ice.parquet"
OUT_AUTHOR_BIPARTITE = PROCESSED_DIR / "ice_bipartite_author_keyword.gexf"
OUT_AUTHOR_PROJECTION = PROCESSED_DIR / "ice_author_projection.gexf"
OUT_GATEKEEPERS = PROCESSED_DIR / "ice_gatekeepers.csv"
OUT_GATEKEEPERS_BY_PERIOD = PROCESSED_DIR / "ice_gatekeepers_by_period.csv"
OUT_SECTION_BIPARTITE = PROCESSED_DIR / "ice_bipartite_section_keyword.gexf"
OUT_SECTION_PROJECTION = PROCESSED_DIR / "ice_section_projection.gexf"
OUT_VOCAB_DIFF = PROCESSED_DIR / "vocab_signed_vs_unsigned.csv"
OUT_INSIGHTS_REPORT = DOCS_DIR / "insights_signed_vs_unsigned.md"

# weighted_projected_graph pesa gia' per numero di keyword condivise -> soglia
# diretta in "keyword condivise". A 1 (nessun filtro) la proiezione autori era
# troppo densa (340 nodi, 38k archi) per leggere la struttura: quasi ogni
# coppia di autori condivide almeno una keyword strutturale di sfondo (es.
# "illegal immigration"). MIN_SHARED_KEYWORDS=3 sfoltisce il rumore.
MIN_SHARED_KEYWORDS = 3

sns.set_theme(style="whitegrid", palette="muted")
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("bipartite_network")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")


## 0. Funzioni condivise (da `topic_network.py`)

`extract_keywords` normalizza le keyword di un articolo (stesso schema break 2025-01 dei notebook precedenti); `get_periods`/`filter_by_period` definiscono le quattro ere politiche usate per il confronto per periodo.

In [ ]:
# tag del sistema editoriale NYT finiti tra le keyword `subject` (es. "internal-open-access-from-nl", "audio-neutral-informative"):
# stessa regex del notebook 02
TECHNICAL_TAG_RE = re.compile(r"^(internal-|content type:|live-detached$|audio-|vis-|your-feed-)")


def extract_keywords(keywords: list) -> list[tuple[str, int, str]]:
    """Da una lista di dict a lista di keyword (value, rank, category) pulite di UN SOLO articolo."""
    NAME_ALIASES = {
        "subject": "subject",
        "organizations": "organizations",
        "organization": "organizations",
        "persons": "persons",
        "person": "persons",
        "glocations": "glocations",
        "location": "glocations",
        "creative_works": "creative_works",
        "title": "creative_works",
    }
    included_names = {"subject", "persons", "glocations", "organizations", "creative_works"}

    ice_string = "immigration and customs enforcement (us)"
    keywords_filtered = []

    for kw in keywords:
        raw_name = kw.get("name", "").strip().lower()
        name = NAME_ALIASES.get(raw_name)

        if name in included_names:
            value = kw.get("value", "").strip().lower()
            # in pochi articoli la stessa keyword compare due volte: contata una sola volta
            if (
                value
                and value != ice_string
                and value not in {k[0] for k in keywords_filtered}
                and not TECHNICAL_TAG_RE.match(value)
            ):
                rank = kw.get("rank", 0)
                keywords_filtered.append((value, rank, name))

    return keywords_filtered


def get_periods(df: pd.DataFrame) -> list[tuple[str, int, int]]:
    """Confini dei periodi da confrontare: (label, start_year incluso, end_year escluso)."""
    max_year = int(df["pub_date"].dt.year.max())
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def filter_by_period(df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    """Filtra df per anno di pubblicazione in [start_year, end_year)."""
    years = df["pub_date"].dt.year
    return df[(years >= start_year) & (years < end_year)]


## 1. Bipartito Autori-Keyword

`extract_authors` normalizza `byline.person` (lista strutturata firstname/middlename/lastname, più robusta del parsing della stringa "By X and Y") in nomi autore per articolo. Bylines redazionali ("By The Editorial Board", "By The New York Times") sono escluse dai nodi Autore — non sono giornalisti individuali.

**Firme del 2025+** (corretto il 2026-10-05): dal 2025 il NYT non compila più `byline.person`, ma la firma è ancora nel testo `byline.original` nel 98% degli articoli ICE 2025+ ("By Hamed Aleaziz", "By Zolan Kanno-Youngs and Hamed Aleaziz"). Prima della correzione questi articoli risultavano senza autore: tutto il 2025-presente spariva dalle reti autori e risultava "unsigned". Ora, quando `person` è vuoto, gli autori si leggono dalla firma testuale: si separa su virgole e "and", si toglie "By" (o "Por" nelle firme in spagnolo) e si scartano le voci che non sono giornalisti (`NON_AUTHOR_RE`: firme istituzionali che iniziano con "The", "Via Youtube", fotografi "Photographs By ..."). Token interamente maiuscolo (artefatto di schema pre~2010, es. "WEBER") sono normalizzati con `.title()`; token già misto-case (es. "McCarthy") restano invariati.

**Firme senza cognome** (corretto il 2026-10-02): in alcuni articoli `byline.person` ha `lastname` vuoto. Michael D. Shear compariva come nodo "Michael" (16 articoli); lo stesso una volta per Scott M. Stringer, Robert M. Morgenthau, Andrew M. Cuomo e Julián Aguilar. Quando manca il cognome, il nome completo si recupera dalla firma testuale `byline.original` ("By Michael D. Shear and Ron Nixon"): si tiene il nome che inizia con il `firstname` registrato, solo se la corrispondenza è unica.

**Keyword uguali al nome di un autore** (escluse dal 2026-10-02, solo nel bipartito Autori): in una parte degli articoli firmati il NYT mette tra le keyword `persons` il nome di chi firma ("preston, julia", "nixon, ron"; anche negli editoriali dei politici, "escobar, veronica"). Identificano l'autore, non un tema, e gonfiavano le keyword condivise tra coautori. Esclusa ogni keyword che inizia con "cognome, nome" di un autore dello stesso insieme di articoli.

Il grafo bipartito ha nodi Autore (`bipartite=0`) e Keyword (`bipartite=1`), arco pesato per numero di articoli in cui autore e keyword coesistono.

**Stesso autore scritto in due modi** tra `byline.person` e firma testuale: "Jonah Engel Bromwich" / "Jonah E. Bromwich", "Orlando Mayorquin" / "Orlando Mayorquín". Uniti con `AUTHOR_ALIASES` (si tiene la forma più usata); `name_variants` segnala nel log eventuali nuovi casi (stesso nome e cognome dopo aver tolto accenti, punti e iniziali).

In [ ]:
def load_ice_articles() -> pd.DataFrame:
    """Carica articles_ice.parquet con keywords/byline decodificati da JSON."""
    df = pd.read_parquet(IN_ICE, columns=["id", "pub_date", "byline_json", "section_name", "keywords_json", "word_count", "web_url"])
    df["byline"] = df["byline_json"].apply(json.loads)
    df["keywords"] = df["keywords_json"].apply(json.loads)
    return df[["id", "pub_date", "byline", "section_name", "keywords", "word_count", "web_url"]]


# voci della firma testuale che non sono giornalisti: firme istituzionali ("The Editorial Board",
# "The New York Times", "The White House, Via Youtube", "The Morning Team") e fotografi
NON_AUTHOR_RE = re.compile(r"^(the |via |photographs by )", re.IGNORECASE)


def byline_names(byline: dict) -> list[str]:
    """Nomi dalla firma testuale: "By A, B and C" -> ["A", "B", "C"] ("Por" per le firme in spagnolo)."""
    text = re.sub(r"^(By|Por)\s+", "", html.unescape(byline.get("original") or "").strip(), flags=re.IGNORECASE)
    return [n.strip() for n in re.split(r",\s*|\s+and\s+", text) if n.strip()]


# stesso giornalista scritto in due modi tra `byline.person` (fino al 2024) e firma testuale (2025+);
# trovati con il controllo sotto (stesso nome e cognome senza accenti e iniziali): si tiene la forma più usata
AUTHOR_ALIASES = {"Jonah Engel Bromwich": "Jonah E. Bromwich", "Orlando Mayorquin": "Orlando Mayorquín"}


def normalize_name(parts: list[str]) -> str:
    name = " ".join(x.title() if x.isupper() else x for x in parts)
    return AUTHOR_ALIASES.get(name, name)


def name_variants(authors) -> pd.DataFrame:
    """Autori diversi con lo stesso nome e cognome dopo aver tolto accenti, punti e iniziali: candidati alias."""
    def key(name):
        tokens = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().lower().replace(".", "").split()
        return (tokens[0], tokens[-1]) if len(tokens) > 1 else (name.lower(), "")
    frame = pd.DataFrame({"author": sorted(set(authors))})
    frame["key"] = frame["author"].map(key)
    return frame[frame["key"].duplicated(keep=False)]


def extract_authors(byline: dict) -> list[str]:
    """Da byline_json a lista di nomi autore normalizzati per UN SOLO articolo.

    Dal 2025 il NYT non compila più `byline.person`: la firma resta solo nel testo `byline.original`.
    """
    persons = byline.get("person") or []
    if not persons:
        names = [normalize_name(n.split()) for n in byline_names(byline) if not NON_AUTHOR_RE.match(n)]
        return list(dict.fromkeys(names))
    authors = []
    for p in persons:
        parts = [p.get("firstname"), p.get("middlename"), p.get("lastname")]
        parts = [x.strip() for x in parts if x and x.strip()]
        if not parts:
            continue
        if not (p.get("lastname") or "").strip():
            matches = [n for n in byline_names(byline) if n.lower().startswith(parts[0].lower())]
            if len(matches) == 1:
                parts = matches[0].split()
        normalized = normalize_name(parts)
        if normalized not in authors:
            authors.append(normalized)
    return authors


def author_name_keywords(authors, keywords) -> set[str]:
    """Keyword che iniziano con 'cognome, nome' di uno degli autori (il NYT tagga a volte chi firma)."""
    keys = {f"{a.split()[-1].lower()}, {a.split()[0].lower()}" for a in authors if len(a.split()) > 1}
    return {k for k in keywords if any(k.startswith(key) for key in keys)}


def build_bipartite_author_keyword(df: pd.DataFrame) -> nx.Graph:
    """Grafo bipartito Autori (bipartite=0) - Keyword (bipartite=1), arco pesato per n. articoli condivisi."""
    df = df.copy()
    df["author"] = df["byline"].apply(extract_authors)
    df["keyword"] = df["keywords"].apply(lambda kws: [kw[0] for kw in extract_keywords(kws)])

    edges = (
        df[["id", "author", "keyword"]]
        .explode("author")
        .explode("keyword")
        .dropna(subset=["author", "keyword"])
    )
    name_keywords = author_name_keywords(edges["author"].unique(), edges["keyword"].unique())
    edges = edges[~edges["keyword"].isin(name_keywords)]

    graph = nx.Graph()
    authors = edges["author"].unique()
    keywords = edges["keyword"].unique()
    graph.add_nodes_from(authors, bipartite=0)
    graph.add_nodes_from(keywords, bipartite=1)

    weights = edges.groupby(["author", "keyword"]).size()
    graph.add_weighted_edges_from((a, k, w) for (a, k), w in weights.items())

    return graph


def export_graph(graph: nx.Graph, path: Path) -> None:
    nx.write_gexf(graph, path)


df = load_ice_articles()
log.info("Costruisco bipartito Autori-Keyword su %d articoli", len(df))
B_author = build_bipartite_author_keyword(df)
n_authors = sum(1 for _, d in B_author.nodes(data=True) if d["bipartite"] == 0)
n_keywords = B_author.number_of_nodes() - n_authors
log.info("Bipartito Autori-Keyword: %d autori, %d keyword, %d archi", n_authors, n_keywords, B_author.number_of_edges())
export_graph(B_author, OUT_AUTHOR_BIPARTITE)
log.info("Scritto %s", OUT_AUTHOR_BIPARTITE.relative_to(BASE_DIR))

variants = name_variants(n for n, d in B_author.nodes(data=True) if d["bipartite"] == 0)
if len(variants):
    log.warning("Possibili autori doppi (aggiungere ad AUTHOR_ALIASES): %s", variants["author"].tolist())


### Proiezione Autore-Autore

`weighted_projected_graph` pesa l'arco per numero di keyword condivise tra due autori. Filtriamo a `min_shared_keywords >= 3` per far emergere solo le coppie con sovrapposizione tematica sostanziale (senza filtro: 340 nodi/38.203 archi, illeggibile).

In [ ]:
def project_author_network(B: nx.Graph, min_shared_keywords: int = 1) -> nx.Graph:
    """Proiezione unipartita Autore-Autore, pesata per numero di keyword condivise."""
    authors = {n for n, d in B.nodes(data=True) if d.get("bipartite") == 0}
    if not authors:
        return nx.Graph()

    graph = bipartite.weighted_projected_graph(B, authors)

    if min_shared_keywords > 1:
        weak_edges = [(u, v) for u, v, w in graph.edges(data="weight") if w < min_shared_keywords]
        graph.remove_edges_from(weak_edges)
        graph.remove_nodes_from(list(nx.isolates(graph)))

    return graph


G_authors = project_author_network(B_author, min_shared_keywords=MIN_SHARED_KEYWORDS)
log.info(
    "Proiezione Autore-Autore (min_shared_keywords=%d): %d nodi, %d archi",
    MIN_SHARED_KEYWORDS, G_authors.number_of_nodes(), G_authors.number_of_edges(),
)
export_graph(G_authors, OUT_AUTHOR_PROJECTION)
log.info("Scritto %s", OUT_AUTHOR_PROJECTION.relative_to(BASE_DIR))


## 2. Gatekeeper editoriali

Top autori per *betweenness centrality* sulla proiezione pesata (peso = keyword condivise → distanza = `1/peso`, più forte il legame più corta la distanza). Tie-break secondario per nome: su pareggi esatti di betweenness l'ordine tra pareggiati dipenderebbe altrimenti dall'iterazione di un `set` Python (hash-randomizzato), non deterministico tra run — osservato in pratica su periodi piccoli.

In [ ]:
def extract_gatekeepers(G_authors: nx.Graph, top_n: int = 10) -> pd.DataFrame:
    """Top N autori per betweenness centrality; scrive anche l'attributo sui nodi di G_authors."""
    for _, _, d in G_authors.edges(data=True):
        d["distance"] = 1.0 / d["weight"] if d["weight"] else float("inf")

    centrality = nx.betweenness_centrality(G_authors, weight="distance")
    nx.set_node_attributes(G_authors, centrality, "betweenness_centrality")

    ranked = sorted(centrality.items(), key=lambda kv: (-kv[1], kv[0]))[:top_n]
    return pd.DataFrame(ranked, columns=["author", "betweenness_centrality"])


gatekeepers = extract_gatekeepers(G_authors, top_n=10)
gatekeepers.to_csv(OUT_GATEKEEPERS, index=False)
log.info("Scritto %s", OUT_GATEKEEPERS.relative_to(BASE_DIR))
gatekeepers


### Gatekeeper per periodo storico

Bipartito + proiezione + gatekeeper ricostruiti da zero per ciascun periodo (non un sotto-grafo del grafo globale: le keyword condivise vanno ricalcolate solo tra articoli dello stesso periodo). Fino al 2026-10-05 il periodo 2025-presente risultava senza gatekeeper, perché gli autori si leggevano solo da `byline.person`, vuoto dal 2025; con la lettura della firma testuale (§1) anche il 2025-presente ha i suoi autori.

In [ ]:
def extract_gatekeepers_by_period(df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    """Gatekeeper per periodo storico (stessi bucket di get_periods)."""
    periods = get_periods(df)
    rows = []

    for label, start, end in periods:
        period_df = filter_by_period(df, start, end)
        B = build_bipartite_author_keyword(period_df)
        G = project_author_network(B, min_shared_keywords=MIN_SHARED_KEYWORDS)
        gk = extract_gatekeepers(G, top_n=top_n)

        gk.insert(0, "period", label)
        gk["n_articles"] = len(period_df)
        gk["n_authors_in_graph"] = G.number_of_nodes()
        rows.append(gk)

        log.info(
            "Periodo %s (%d articoli, grafo %d nodi/%d archi): top %d gatekeeper -> %s",
            label, len(period_df), G.number_of_nodes(), G.number_of_edges(), top_n,
            ", ".join(gk["author"]) if not gk.empty else "nessuno (grafo vuoto)",
        )

    return pd.concat(rows, ignore_index=True)


gatekeepers_by_period = extract_gatekeepers_by_period(df, top_n=3)
gatekeepers_by_period.to_csv(OUT_GATEKEEPERS_BY_PERIOD, index=False)
log.info("Scritto %s", OUT_GATEKEEPERS_BY_PERIOD.relative_to(BASE_DIR))
gatekeepers_by_period


## 3. Bipartito Sezione-Keyword

Stessa logica del bipartito Autori, con `section_name` (già una colonna flat) al posto degli autori — nessun filtro di soglia sulla proiezione.

In [ ]:
def build_bipartite_section_keyword(df: pd.DataFrame) -> nx.Graph:
    """Grafo bipartito Sezione (bipartite=0) - Keyword (bipartite=1)."""
    df = df.copy()
    df["keyword"] = df["keywords"].apply(lambda kws: [kw[0] for kw in extract_keywords(kws)])

    edges = (
        df[["id", "section_name", "keyword"]]
        .explode("keyword")
        .dropna(subset=["section_name", "keyword"])
    )

    graph = nx.Graph()
    sections = edges["section_name"].unique()
    keywords = edges["keyword"].unique()
    graph.add_nodes_from(sections, bipartite=0)
    graph.add_nodes_from(keywords, bipartite=1)

    weights = edges.groupby(["section_name", "keyword"]).size()
    graph.add_weighted_edges_from((s, k, w) for (s, k), w in weights.items())

    return graph


def project_section_network(B: nx.Graph) -> nx.Graph:
    """Proiezione unipartita Sezione-Sezione."""
    sections = {n for n, d in B.nodes(data=True) if d.get("bipartite") == 0}
    return bipartite.weighted_projected_graph(B, sections)


log.info("Costruisco bipartito Sezione-Keyword su %d articoli", len(df))
B_section = build_bipartite_section_keyword(df)
export_graph(B_section, OUT_SECTION_BIPARTITE)
log.info("Scritto %s", OUT_SECTION_BIPARTITE.relative_to(BASE_DIR))

G_sections = project_section_network(B_section)
export_graph(G_sections, OUT_SECTION_PROJECTION)
log.info(
    "Scritto %s (%d nodi, %d archi)",
    OUT_SECTION_PROJECTION.relative_to(BASE_DIR), G_sections.number_of_nodes(), G_sections.number_of_edges(),
)


## 4. Grafici sui gatekeeper

Bar chart dei top gatekeeper, scatter volume-vs-influenza, heatmap gatekeeper×keyword, bar chart per periodo — tutti su `G_authors` (già arricchito dell'attributo `betweenness_centrality` da `extract_gatekeepers` sopra) e `df`.

In [ ]:
def plot_gatekeepers_barchart(G_authors: nx.Graph, top_n: int = 15, output_dir: Path = PLOTS_DIR) -> Path:
    """Bar chart orizzontale dei top N autori per betweenness centrality."""
    centrality = nx.get_node_attributes(G_authors, "betweenness_centrality")
    top = (
        pd.Series(centrality, name="betweenness_centrality")
        .sort_values(ascending=False)
        .head(top_n)
        .rename_axis("author")
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(9, 0.4 * len(top) + 1.5))
    sns.barplot(data=top, y="author", x="betweenness_centrality", order=top["author"], ax=ax)
    ax.set_xlabel("Betweenness centrality")
    ax.set_ylabel("")
    ax.set_title(f"NYT ICE — Top {top_n} gatekeeper editoriali per betweenness centrality")
    sns.despine(left=True, bottom=True)

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "gatekeepers_barchart.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


out = plot_gatekeepers_barchart(G_authors, top_n=15)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))


In [ ]:
def author_article_counts(df: pd.DataFrame) -> pd.Series:
    """Numero di articoli ICE per autore (un articolo con N coautori conta 1 per ciascuno)."""
    exploded = df.assign(author=df["byline"].apply(extract_authors))[["id", "author"]].explode("author")
    return exploded.dropna(subset=["author"]).groupby("author")["id"].nunique()


def plot_volume_vs_influence(G_authors: nx.Graph, df: pd.DataFrame, output_dir: Path = PLOTS_DIR) -> Path:
    """Scatter: volume articoli vs betweenness centrality, un punto per autore. Etichette solo per gli outlier."""
    centrality = pd.Series(nx.get_node_attributes(G_authors, "betweenness_centrality"), name="betweenness_centrality")
    volume = author_article_counts(df).rename("n_articles")

    frame = pd.concat([volume, centrality], axis=1, join="inner").rename_axis("author").reset_index()
    outliers = set(frame.nlargest(5, "n_articles")["author"]) | set(frame.nlargest(5, "betweenness_centrality")["author"])

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=frame, x="n_articles", y="betweenness_centrality", ax=ax, s=60, alpha=0.7, edgecolor="black", linewidth=0.4)

    for _, row in frame[frame["author"].isin(outliers)].iterrows():
        ax.annotate(row["author"], (row["n_articles"], row["betweenness_centrality"]), xytext=(6, 3), textcoords="offset points", fontsize=8)

    ax.set_xlabel("Numero di articoli ICE scritti")
    ax.set_ylabel("Betweenness centrality")
    ax.set_title("NYT ICE — Volume di produzione vs influenza strutturale (gatekeeping)")
    sns.despine()

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "volume_vs_influence.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


out = plot_volume_vs_influence(G_authors, df)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))


In [ ]:
def top_keywords_overall(df: pd.DataFrame, top_n: int) -> list[str]:
    """Le top N keyword (per conteggio articoli) sull'intero corpus df."""
    keyword_lists = df["keywords"].apply(lambda kws: [kw[0] for kw in extract_keywords(kws)])
    counts = keyword_lists.explode().value_counts()
    return counts.head(top_n).index.tolist()


def plot_gatekeeper_heatmap(df, G_authors, top_gatekeepers_n=5, top_keywords_n=10, output_dir=PLOTS_DIR):
    """Heatmap gatekeeper x keyword: n. articoli in cui ciascun top gatekeeper ha usato ciascuna top keyword."""
    centrality = pd.Series(nx.get_node_attributes(G_authors, "betweenness_centrality"))
    top_gatekeepers = centrality.sort_values(ascending=False).head(top_gatekeepers_n).index.tolist()
    top_keywords = top_keywords_overall(df, top_keywords_n)

    work = df.assign(
        author=df["byline"].apply(extract_authors),
        keyword=df["keywords"].apply(lambda kws: [kw[0] for kw in extract_keywords(kws)]),
    )[["id", "author", "keyword"]].explode("author").explode("keyword")

    work = work[work["author"].isin(top_gatekeepers) & work["keyword"].isin(top_keywords)]
    pivot = (
        work.groupby(["author", "keyword"])["id"].nunique()
        .unstack(fill_value=0)
        .reindex(index=top_gatekeepers, columns=top_keywords, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(1.1 * top_keywords_n + 2, 0.6 * top_gatekeepers_n + 2))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax, cbar_kws={"label": "N. articoli"})
    ax.set_xlabel("Keyword")
    ax.set_ylabel("Gatekeeper")
    ax.set_title(f"NYT ICE — Top {top_gatekeepers_n} gatekeeper x top {top_keywords_n} keyword")
    plt.setp(ax.get_xticklabels(), rotation=40, ha="right")

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "gatekeeper_heatmap.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


out = plot_gatekeeper_heatmap(df, G_authors, top_gatekeepers_n=5, top_keywords_n=10)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))


In [ ]:
def plot_gatekeepers_by_period(df: pd.DataFrame, top_n: int = 3, output_dir: Path = PLOTS_DIR) -> Path:
    """Bar chart per periodo storico: top N gatekeeper e betweenness, un subplot per era (assi non condivisi)."""
    periods = get_periods(df)
    by_period = extract_gatekeepers_by_period(df, top_n=top_n)

    fig, axes = plt.subplots(1, len(periods), figsize=(4.3 * len(periods), 4.2))

    for ax, (label, start, end) in zip(axes, periods):
        sub = by_period[by_period["period"] == label]
        n_articles = len(filter_by_period(df, start, end))

        if sub.empty:
            ax.text(0.5, 0.5, "nessun gatekeeper\n(bylines mancanti)", ha="center", va="center", transform=ax.transAxes, fontsize=9, color="gray")
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            sns.barplot(data=sub, y="author", x="betweenness_centrality", order=sub["author"], ax=ax)
            ax.set_xlabel("Betweenness")
            ax.set_ylabel("")

        ax.set_title(f"{label}\n(n={n_articles} articoli)", fontsize=10)

    fig.suptitle(f"NYT ICE — Top {top_n} gatekeeper per periodo storico", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "gatekeepers_by_period.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


out = plot_gatekeepers_by_period(df, top_n=3)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))


## 5. Articoli signed vs unsigned: vocabolario e lunghezza

Ipotesi: la cronaca unsigned (senza byline individuale) usa vocabolario operativo/reattivo (arresti, raid), quella signed vocabolario sistemico/politico (diritti, riforme). Per ciascun gruppo calcoliamo la frequenza normalizzata di ogni keyword (% di articoli del gruppo in cui compare — non il conteggio assoluto, che premierebbe sempre il gruppo più numeroso); lo scarto `diff_pct = freq_signed − freq_unsigned` misura la polarizzazione. Soglia minima `min_freq=10` (somma tra i due gruppi) per escludere keyword rare dal confronto.

**Definizione** (corretta il 2026-10-05): signed = almeno un giornalista individuale nella firma, letto da `byline.person` o, dal 2025, dalla firma testuale (§1). Unsigned = nessuna firma o solo firme istituzionali ("The Editorial Board", "The New York Times"). Con la definizione precedente tutti i 1.224 articoli 2025+ risultavano unsigned e il confronto era in realtà "prima del 2025 contro 2025+".

In [ ]:
def compare_vocabularies(df: pd.DataFrame, min_freq: int = 10) -> pd.DataFrame:
    """Confronta il vocabolario delle keyword tra articoli signed e unsigned. Richiede has_byline (bool) e keywords (tuple list)."""
    n_signed = int(df["has_byline"].sum())
    n_unsigned = int((~df["has_byline"]).sum())

    work = df[["has_byline", "keywords"]].copy()
    work["keyword"] = work["keywords"].apply(lambda kws: [kw[0] for kw in kws])
    flat = work[["has_byline", "keyword"]].explode("keyword").dropna(subset=["keyword"])

    counts = flat.groupby(["has_byline", "keyword"]).size().unstack("has_byline", fill_value=0)
    counts = counts.rename(columns={True: "n_signed", False: "n_unsigned"})
    for col in ("n_signed", "n_unsigned"):
        if col not in counts.columns:
            counts[col] = 0

    counts = counts[(counts["n_signed"] + counts["n_unsigned"]) >= min_freq]

    result = counts.rename_axis("keyword").reset_index()[["keyword", "n_signed", "n_unsigned"]]
    result["freq_signed_pct"] = result["n_signed"] / n_signed * 100 if n_signed else 0.0
    result["freq_unsigned_pct"] = result["n_unsigned"] / n_unsigned * 100 if n_unsigned else 0.0
    result["diff_pct"] = result["freq_signed_pct"] - result["freq_unsigned_pct"]

    return result.sort_values("diff_pct", ascending=False).reset_index(drop=True)


df_vocab = load_ice_articles()
df_vocab["has_byline"] = df_vocab["byline"].apply(lambda b: bool(extract_authors(b)))
df_vocab["keywords"] = df_vocab["keywords"].apply(extract_keywords)

n_signed = int(df_vocab["has_byline"].sum())
n_unsigned = len(df_vocab) - n_signed
log.info("Corpus: %d signed, %d unsigned (%.1f%% unsigned)", n_signed, n_unsigned, 100 * n_unsigned / len(df_vocab))

vocab_diff = compare_vocabularies(df_vocab, min_freq=10)
vocab_diff.to_csv(OUT_VOCAB_DIFF, index=False)
log.info("Scritto %s (%d keyword sopra soglia)", OUT_VOCAB_DIFF.relative_to(BASE_DIR), len(vocab_diff))
vocab_diff.head(10)


In [ ]:
def plot_diverging_bars(vocab_diff_df: pd.DataFrame, top_n: int = 15, output_dir: Path = PLOTS_DIR) -> Path:
    """Diverging bar chart orizzontale delle keyword piu' polarizzate (top N signed + top N unsigned)."""
    top_signed = vocab_diff_df.nlargest(top_n, "diff_pct")
    top_unsigned = vocab_diff_df.nsmallest(top_n, "diff_pct")

    plot_df = pd.concat([top_unsigned, top_signed]).sort_values("diff_pct").reset_index(drop=True)
    colors = ["#c0392b" if v < 0 else "#2980b9" for v in plot_df["diff_pct"]]

    fig, ax = plt.subplots(figsize=(9, 0.4 * len(plot_df) + 1.5))
    ax.barh(plot_df["keyword"], plot_df["diff_pct"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Scarto % (Signed − Unsigned)")
    ax.set_ylabel("")
    ax.set_title(f"Vocabolario ICE: top {top_n} keyword polarizzate — Signed vs Unsigned")
    sns.despine(left=True, bottom=True)

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "vocab_diverging_bars.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


out = plot_diverging_bars(vocab_diff, top_n=15)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))


In [ ]:
def generate_insights_report(vocab_diff_df: pd.DataFrame, output_dir: Path = DOCS_DIR, top_n: int = 15) -> Path:
    """Genera insights_signed_vs_unsigned.md con le top N keyword per gruppo."""
    top_signed = vocab_diff_df.nlargest(top_n, "diff_pct")
    top_unsigned = vocab_diff_df.nsmallest(top_n, "diff_pct").sort_values("diff_pct")

    def to_md_table(rows: pd.DataFrame) -> str:
        lines = ["| Keyword | Signed % | Unsigned % | Scarto % |", "|---|---|---|---|"]
        for _, r in rows.iterrows():
            lines.append(f"| {r['keyword']} | {r['freq_signed_pct']:.1f} | {r['freq_unsigned_pct']:.1f} | {r['diff_pct']:+.1f} |")
        return "\n".join(lines)

    content = f"""# Vocabolario ICE: Signed vs Unsigned

Confronto tra il vocabolario tematico (keyword NYT) degli articoli con byline
individuale (signed) e quelli senza (unsigned), sul corpus `articles_ice`.

## Top {top_n} keyword associate al giornalismo signed

{to_md_table(top_signed)}

## Top {top_n} keyword associate al giornalismo unsigned

{to_md_table(top_unsigned)}
"""

    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "insights_signed_vs_unsigned.md"
    out_path.write_text(content, encoding="utf-8")
    return out_path


report_path = generate_insights_report(vocab_diff, top_n=15)
log.info("Scritto %s", report_path.relative_to(BASE_DIR))
print(report_path.read_text(encoding="utf-8")[:800])


**Come leggere il grafico del vocabolario.** Barre verso destra (blu): keyword più frequenti tra gli articoli signed; verso sinistra (rosso): più frequenti tra gli unsigned. La lunghezza è lo scarto in punti percentuali tra le quote di articoli dei due gruppi.

**Risultato** (con la definizione corretta del 2026-10-05). Gli unsigned si distinguono per la keyword `editorials` (24% contro 0%) e per keyword dell'epoca Obama ("illegal immigrants", Obama, Napolitano, John Morton): sono gli editoriali, concentrati nel 2003-2016 (59 dei 120 unsigned). I signed hanno più keyword di cronaca degli ultimi anni (Trump, DHS, azioni federali nelle città, Chicago, proteste). **Attenzione**: anche così il confronto resta in parte un confronto tra periodi, perché metà degli unsigned è del 2003-2016; e il gruppo unsigned è piccolo (120 articoli), quindi le sue quote sono instabili. L'ipotesi iniziale (unsigned = cronaca operativa) non è confermata: gli unsigned sono soprattutto commento.

### 5.1 Lunghezza degli articoli signed e unsigned

Domanda: gli articoli unsigned sono notizie brevi (flash news) o articoli completi? Si usa `word_count`, la lunghezza dell'**articolo intero** registrata dal NYT (non solo titolo e abstract). Gli articoli con `word_count` = 0 (pagine interattive, video, alcuni live) sono esclusi dalla lunghezza.

Il confronto si fa anche **per periodo**, perché la lunghezza media degli articoli cambia nel tempo e i due gruppi non sono distribuiti allo stesso modo tra i periodi. A destra, dove stanno gli unsigned: la sezione si legge dall'URL (`/opinion/`, `/us/`, `/interactive/`...).

In [ ]:
SIGNED_COLORS = {True: "#2a78d6", False: "#eb6834"}
SIGNED_NAMES = {True: "signed", False: "unsigned"}


def length_by_group(df: pd.DataFrame) -> pd.DataFrame:
    """Lunghezza (word_count dell'articolo intero) per gruppo e periodo; esclusi gli articoli con word_count = 0."""
    work = df[df["word_count"] > 0].copy()
    labels = {label: (start, end) for label, start, end in get_periods(df)}
    work["period"] = work["pub_date"].dt.year.map(lambda y: next(l for l, (s, e) in labels.items() if s <= y < e))
    return work


def plot_length_signed_unsigned(work: pd.DataFrame, df: pd.DataFrame, output_dir: Path = PLOTS_DIR) -> Path:
    """A sinistra: distribuzione della lunghezza per periodo, signed contro unsigned. A destra: sezioni degli unsigned."""
    periods = [label for label, _, _ in get_periods(df)] + ["all periods"]
    fig, (ax, ax_sec) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2.4, 1], "wspace": 0.35})

    for i, period in enumerate(periods):
        sub = work if period == "all periods" else work[work["period"] == period]
        for offset, signed in [(-0.18, True), (0.18, False)]:
            values = sub.loc[sub["has_byline"] == signed, "word_count"]
            if values.empty:
                continue
            ax.boxplot(values, positions=[i + offset], widths=0.3, showfliers=False, patch_artist=True,
                       boxprops={"facecolor": SIGNED_COLORS[signed], "edgecolor": "white"},
                       medianprops={"color": "white", "linewidth": 2},
                       whiskerprops={"color": SIGNED_COLORS[signed]}, capprops={"color": SIGNED_COLORS[signed]})
            q1, q3 = values.quantile([0.25, 0.75])
            top = values[values <= q3 + 1.5 * (q3 - q1)].max()
            ax.text(i + offset, top + 40, f"{values.median():.0f}\nn={len(values)}",
                    ha="center", va="bottom", fontsize=7, color="#52514e")
    ax.set_xticks(range(len(periods)))
    ax.set_xticklabels([p.replace("presente", "present") for p in periods])
    ax.axvline(len(periods) - 1.5, color="#c3c2b7", linewidth=0.8, linestyle=":")
    ax.set_ylim(top=ax.get_ylim()[1] * 1.12)
    ax.set_ylabel("article length (words, full text)")
    ax.set_title("NYT ICE articles — length of signed vs unsigned articles\n"
                 "(boxes: 25th-75th percentile; label: median and number of articles; outliers not shown)",
                 loc="left", fontsize=10)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in SIGNED_COLORS.values()],
              labels=list(SIGNED_NAMES.values()), loc="upper left", frameon=False)

    unsigned = df[~df["has_byline"]]
    kind = unsigned["web_url"].str.extract(r"nytimes\.com/(?:\d{4}/\d\d/\d\d/)?([a-z-]+)/")[0].fillna("other")
    counts = kind.value_counts()
    counts = pd.concat([counts.head(5), pd.Series({"other": counts.iloc[5:].sum()})]) if len(counts) > 5 else counts
    counts = counts[counts > 0].sort_values()
    ax_sec.barh(counts.index, counts.values, color=SIGNED_COLORS[False], height=0.6)
    for y, v in enumerate(counts.values):
        ax_sec.text(v + 1, y, str(v), va="center", fontsize=8, color="#52514e")
    ax_sec.set_xlabel("unsigned articles")
    ax_sec.set_title(f"Where the {len(unsigned)} unsigned articles are\n(section in the article URL)", loc="left", fontsize=10)

    for a in (ax, ax_sec):
        a.grid(False)
        a.grid(axis="y" if a is ax else "x", alpha=0.4)
        sns.despine(ax=a)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / "length_signed_unsigned.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


length_work = length_by_group(df_vocab)
log.info("Articoli con word_count = 0 esclusi dalla lunghezza: %d", (df_vocab["word_count"] == 0).sum())
log.info("Lunghezza mediana: signed %.0f parole, unsigned %.0f parole",
         length_work.loc[length_work["has_byline"], "word_count"].median(),
         length_work.loc[~length_work["has_byline"], "word_count"].median())
out = plot_length_signed_unsigned(length_work, df_vocab)
log.info("Scritto %s", out.relative_to(BASE_DIR))
display(Image(filename=str(out)))

**Come leggere il grafico.** A sinistra, per ogni periodo, due box: blu signed, arancione unsigned. Il box va dal 25° al 75° percentile della lunghezza, la riga bianca è la mediana; sopra il box, la mediana in parole e il numero di articoli. L'ultima coppia, separata, è su tutti i periodi. A destra, le sezioni in cui stanno gli articoli unsigned.

**Risultato.**

- **Gli unsigned sono pochi**: 120 articoli su 2.093 (5,7%), di cui 59 nel 2003-2016, 26 nel 2017-2020, 5 nel 2021-2024 e 30 nel 2025+.
- **Sono più corti, ma non sono flash news.** Mediana 542 parole contro 976 dei signed; in ogni periodo con abbastanza articoli gli unsigned sono più corti (2003-2016: 416 contro 756; 2017-2020: 481 contro 1.060; 2025+: 744 contro 985). Ma 93 su 120 stanno nella sezione **Opinion**: sono soprattutto **editoriali** ("By The Editorial Board"), testi da 400-700 parole con una posizione argomentata, non notizie brevi d'agenzia. Il resto sono pagine interattive (10), podcast (6) e video (2), che non sono articoli di cronaca.
- **Notizie brevi senza firma praticamente non ci sono**: solo 5 unsigned nella sezione U.S. Il NYT firma quasi sempre la cronaca su ICE, anche quella breve.
- Il 2021-2024 ha solo 2 unsigned con lunghezza: quella coppia di box non si legge.

## Riepilogo

Output scritti da questo notebook (stessi path/nomi già prodotti da `bipartite_network.py`, `bipartite_plots.py`, `vocabulary_analysis.py`):

- `ice_bipartite_author_keyword.gexf`, `ice_author_projection.gexf`, `ice_gatekeepers.csv`, `ice_gatekeepers_by_period.csv`
- `ice_bipartite_section_keyword.gexf`, `ice_section_projection.gexf`
- `plots/{gatekeepers_barchart,volume_vs_influence,gatekeeper_heatmap,gatekeepers_by_period,vocab_diverging_bars}.png`
- `vocab_signed_vs_unsigned.csv`, `docs/insights_signed_vs_unsigned.md`
